## 1. Setup and Imports

In [1]:
import sys
project_root = "/Users/sanedara/Documents/Machine_learning_practise/recommendation_system"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.baseline import baseline_recommender
from src.data.loader import MovieLensLoader
import pandas as pd


## 2. Load Data

Load ratings, movies, and use train/test split (u1.base and u1.test)

In [2]:
baseline_model = baseline_recommender()
loader = MovieLensLoader()
ratings = loader.load_ratings()
movies = loader.load_movies()
print(ratings.head(), movies.head())

/Users/sanedara/Documents/Machine_learning_practise/recommendation_system/src/data/../../data/ml-100k/u.data
/Users/sanedara/Documents/Machine_learning_practise/recommendation_system/src/data/../../data/ml-100k/u.item
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596    movie_id              title release_date  video_release_date  \
0         1   Toy Story (1995)  01-Jan-1995                 NaN   
1         2   GoldenEye (1995)  01-Jan-1995                 NaN   
2         3  Four Rooms (1995)  01-Jan-1995                 NaN   
3         4  Get Shorty (1995)  01-Jan-1995                 NaN   
4         5     Copycat (1995)  01-Jan-1995                 NaN   

                                            IMDb_URL  unknown  Action  \
0  http://us.imdb.com/M/title-exact?Toy%20Story%2...        0       0   

## 3. Base Recommender Implementation

Strategy:
1. build a smart recommendation system with dumb global popularity algorithm

In [ ]:


def split_user(df, frac=0.85):
    df = df.sort_values("timestamp")
    cut = int(frac * len(df))
    return df.iloc[:cut], df.iloc[cut:]

train_parts = []
test_parts = []

for uid, df_u in ratings.groupby("user_id"):
    if len(df_u) < 15:
        continue
    tr, te = split_user(df_u, 0.85)
    if len(tr) == 0 or len(te) == 0:
        continue
    train_parts.append(tr)
    test_parts.append(te)

train_split = pd.concat(train_parts, ignore_index=True)
test_split  = pd.concat(test_parts, ignore_index=True)

pop_rank = (train_split[train_split['rating'] >= 4].groupby('item_id').size().reset_index(name='pop_rank'))
pop_rank = pop_rank.sort_values(["pop_rank", "item_id"], ascending=[False, True]).reset_index(drop=True)

train_seen = train_split.groupby('user_id')['item_id'].apply(set).to_dict()


def recommend_movies(user_id):
    if user_id not in train_seen.keys():
        return []
    seen_movies = train_seen[user_id]
    top_k = []
    for i in range(len(pop_rank)):
        if pop_rank.iloc[i]['item_id'] not in seen_movies:
            top_k.append(pop_rank.iloc[i]['item_id'])
        if len(top_k) == 12:
            break
    return top_k

suggested_movies = recommend_movies(10)
id_to_title = dict(zip(movies['movie_id'], movies['title']))
movie_names = []
for movie_id in suggested_movies:
    movie_names.append(id_to_title[movie_id])
print(movie_names)
            


[180    Return of the Jedi (1983)
Name: title, dtype: object, 257    Contact (1997)
Name: title, dtype: object, 171    Empire Strikes Back, The (1980)
Name: title, dtype: object, 6    Twelve Monkeys (1995)
Name: title, dtype: object, 299    Air Force One (1997)
Name: title, dtype: object, 78    Fugitive, The (1993)
Name: title, dtype: object, 317    Schindler's List (1993)
Name: title, dtype: object, 172    Princess Bride, The (1987)
Name: title, dtype: object, 287    Scream (1996)
Name: title, dtype: object, 312    Titanic (1997)
Name: title, dtype: object, 236    Jerry Maguire (1996)
Name: title, dtype: object, 116    Rock, The (1996)
Name: title, dtype: object]
